# 03. Ephemeris and O-C Analysis

Fit a linear transit ephemeris `T(E) = T0 + E * P` to a set of measured
transit-center times and build the Observed-minus-Calculated (O-C)
diagram. A sloped or curved O-C means the true period differs from the
reference or varies with time (a transit-timing variation, TTV).

This notebook uses the **project's own fitter**
(`src/muscat_db/ephemeris_math.py`), the same weighted least-squares
routine behind the web O-C tool, so the numbers match the app.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Load ephemeris_math.py directly by file. It is pure standard library,
# so this avoids importing the full muscat_db package (which needs extra
# dependencies not present in the prose env).
src = Path("../src/muscat_db/ephemeris_math.py").resolve()
spec = importlib.util.spec_from_file_location("ephemeris_math", src)
ephemeris_math = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ephemeris_math)
fit_linear_ephemeris = ephemeris_math.fit_linear_ephemeris
print("loaded fitter from", src)

## 1. Transit-center measurements

In production these come from per-epoch transit fits (`transit_fit.py` /
the `timer` package): each observed transit yields a center time `Tc` in
BJD_TDB with an uncertainty. Here we synthesize a self-contained set: a
reference ephemeris, per-point timing scatter, and a small transit-timing
variation (TTV) so the O-C diagram has visible structure.

The TTV phase is chosen to be symmetric about the middle epoch. That keeps
it from mimicking a linear trend, so the O-C wave is clearly a *curvature*
and not just a wrong period.

In [ ]:
REF_T0 = 2456950.81156   # BJD_TDB
REF_PERIOD = 0.9255462   # days

rng = np.random.default_rng(7)
epochs = np.arange(0, 49)
e_center = (epochs[0] + epochs[-1]) / 2

ttv_amp_min = 2.0        # minutes
ttv_cycle = 16           # epochs per TTV cycle

# Reference ephemeris + a TTV that is even (symmetric) about the mid-epoch.
ttv = (ttv_amp_min / 1440.0) * np.cos(2 * np.pi * (epochs - e_center) / ttv_cycle)
true_tc = REF_T0 + epochs * REF_PERIOD + ttv

uncs = rng.uniform(1e-4, 3e-4, size=epochs.size)   # ~10-25 s
obs_tc = true_tc + rng.normal(0, uncs)

print(f"{epochs.size} synthetic transit centers, median uncertainty {np.median(uncs)*1440:.2f} min")

## 2. Fit the linear ephemeris

`fit_linear_ephemeris` centers the epochs for numerical stability, does a
weighted least-squares fit (weights `1/unc**2`), and propagates the
covariance back to `E = 0`. It returns best-fit `T0` and `P` with formal
uncertainties.

Note the caveat those formal uncertainties carry: they assume the scatter
about the line is pure measurement noise. A real TTV violates that, so when
one is present the honest period uncertainty is larger than the formal
value. The O-C diagram below is what reveals the TTV.

In [ ]:
fit = fit_linear_ephemeris(
    epochs.tolist(),
    obs_tc.tolist(),
    uncs.tolist(),
    REF_T0,
    REF_PERIOD,
    fit_method="weighted",
)

print(f"fit succeeded : {fit['was_fit']}")
print(f"T0 = {fit['t0_fit']:.6f} +/- {fit['t0_fit_unc']:.6f} BJD_TDB")
print(f"P  = {fit['period_fit']:.7f} +/- {fit['period_fit_unc']:.7f} d")

## 3. The O-C diagram

O-C is the observed center minus the center predicted by the *fitted*
linear ephemeris, in minutes. With the straight-line term removed, any
remaining structure is the non-linear signal, here the TTV.

In [ ]:
calc_tc = fit["t0_fit"] + epochs * fit["period_fit"]
oc_min = (obs_tc - calc_tc) * 1440.0
oc_err_min = uncs * 1440.0

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(epochs, oc_min, yerr=oc_err_min, fmt="o", ms=5, capsize=2, color="C0")
ax.axhline(0, color="C3", ls="--", lw=1)
ax.set_xlabel("Epoch E")
ax.set_ylabel("O - C (minutes)")
_ = ax.set_title("Observed minus calculated transit times")
fig.tight_layout()
plt.show()

## 4. Reading the diagram

Three shapes to recognize:

- **Flat, scattered around zero:** the linear ephemeris is adequate; the
  period is constant within the errors.
- **Straight slope:** the *assumed* period is slightly wrong. Refitting
  removes the slope (which is why the plot above, drawn against the fitted
  ephemeris, is not tilted).
- **Curve or wave:** a genuine TTV, e.g. from a perturbing planet or
  orbital decay. The `harmonic` package (`ttv_fit.py`) models these.

The residual scatter should exceed the timing errors only when real
structure is present:

In [ ]:
resid_rms = np.sqrt(np.mean(oc_min ** 2))
print(f"O-C RMS               : {resid_rms:.2f} min")
print(f"median timing error   : {np.median(oc_err_min):.2f} min")
if resid_rms > 2 * np.median(oc_err_min):
    print("-> residual structure exceeds the measurement noise (real signal)")
else:
    print("-> residuals consistent with noise (no TTV detected)")

## 5. What a wrong period looks like

To make the "slope means wrong period" rule concrete, take the *same*
transit centers and compute O-C against a period deliberately set 5 s per
epoch too long. No randomness here, just geometry: a wrong period adds a
clean linear tilt on top of the TTV.

In [ ]:
wrong_period = fit["period_fit"] + 5.0 / 86400.0   # +5 s per epoch
oc_wrong = (obs_tc - (fit["t0_fit"] + epochs * wrong_period)) * 1440.0

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(epochs, oc_min, yerr=oc_err_min, fmt="o", ms=5, capsize=2,
            color="C0", label="fitted period (TTV only)")
ax.plot(epochs, oc_wrong, "s-", ms=4, color="C1", alpha=0.7,
        label="period +5 s/epoch (adds a slope)")
ax.axhline(0, color="C3", ls="--", lw=1)
ax.set_xlabel("Epoch E")
ax.set_ylabel("O - C (minutes)")
_ = ax.set_title("A wrong period tilts the O-C diagram")
ax.legend()
fig.tight_layout()
plt.show()

## Summary

- A linear ephemeris is two numbers, `T0` and `P`, fit to transit-center
  times.
- The O-C diagram removes that linear trend so residual timing structure
  stands out: a slope means the assumed period is off, a wave means a TTV.
- Formal period uncertainties from a linear fit assume pure measurement
  noise; a real TTV makes the true uncertainty larger.
- This notebook called the exact fitter used by the web O-C tool, so the
  results are reproducible against the production app.
- For TTV modeling beyond a straight line, see `ttv_fit.py` (`harmonic`).